# Seleccion de Variables con Step Forward


En el presente notebook se generan y se evalúan los siguientes modelos.

* Regresión lineal
* Regresion lineal regularizada
* Árbol de regresión
* KNN para regresión
* Regresión no paramétrica

El preprocesamiento principal se realiza mediante una seleccion de variables utilizando step forwarding para regresion lineal. 

In [1]:
library(MASS)
library(leaps)
library(glmnet)
library(pracma)
source("Utils.R")

Loading required package: Matrix
Loading required package: foreach
Loaded glmnet 2.0-16


Attaching package: ‘pracma’

The following objects are masked from ‘package:Matrix’:

    expm, lu, tril, triu



In [2]:
library(reticulate)
np <- import("numpy", convert = FALSE)
pd <- import("pandas", convert = FALSE)

StScaler <- import("sklearn.preprocessing", convert = FALSE)$StandardScaler
MMScaler <- import("sklearn.preprocessing", convert = FALSE)$MinMaxScaler

In [3]:
setSeed()

vasijas_X <- obtener_X()
vasijas_Y <- obtener_Y()

df <- vasijas_X
df$Y <- vasijas_Y

hold_out_ind <- obtener_holdout_ind(vasijas_X)

HOLDOUT_X <- vasijas_X[hold_out_ind, ]
HOLDOUT_Y <- vasijas_Y[hold_out_ind]
HOLDOUT_DF <- df[hold_out_ind, ]

TRAIN_Y <- vasijas_Y[-hold_out_ind]
TRAIN_X <- vasijas_X[-hold_out_ind, ]
TRAIN_DF <- df[-hold_out_ind, ]

# Step Forward con graficos filtrando primero 140 variables

In [4]:
n = nrow(TRAIN_X)

forw <- regsubsets(Y~., data = TRAIN_DF, method = "forward", nvmax = 140)

Warning message in leaps.setup(x, y, wt = wt, nbest = nbest, nvmax = nvmax, force.in = force.in, :
“142  linear dependencies found”

In [5]:
SELECTED_TRAIN_X <- TRAIN_X[, names(which(summary(forw)$which[140, ]))[2:(141)]]
SELECTED_TRAIN_DF <- SELECTED_TRAIN_X
SELECTED_TRAIN_DF$Y <- TRAIN_Y

SELECTED_HOLDOUT_X <- HOLDOUT_X[, names(which(summary(forw)$which[140, ]))[2:(141)]]
SELECTED_HOLDOUT_DF <- SELECTED_HOLDOUT_X
SELECTED_HOLDOUT_DF$Y <- HOLDOUT_Y

### Utilizamos el metodo Forward (utiliza F-Score para elegir variables)

In [6]:
# http://www.science.smith.edu/~jcrouser/SDS293/labs/lab9-r.html

predict.regsubsets = function(object,newdata,id,...){
      form = as.formula(object$call[[2]]) # Extract the formula used when we called regsubsets()
      mat = model.matrix(form,newdata)    # Build the model matrix
      coefi = coef(object,id=id)          # Extract the coefficiants of the ith model
      xvars = names(coefi)                # Pull out the names of the predictors used in the ith model
      mat[,xvars]%*%coefi               # Make predictions using matrix multiplication
}

In [7]:
setSeed()

n = nrow(SELECTED_TRAIN_X)
k = 20

nvmax = 140-n/k
resultados <- matrix(0, nrow = 1, ncol = nvmax)

X = SELECTED_TRAIN_X

indices = obtener_indices_kfold(n, k)

for(i in indices){
    DF_train = X[-i,]
    DF_train$Y = TRAIN_Y[-i]

    DF_test = X[i,]
    DF_test$Y = TRAIN_Y[i]
    
    forw <- regsubsets(Y~., data =DF_train, method = "forward", nvmax=nvmax)
    
    for(j in 1:nvmax){
        Y_pred = predict(forw, DF_test, id=j)
        resultados[[1, j]] = resultados[[1, j]] + sum((DF_test$Y - Y_pred)^2)
    }
}

In [9]:
resultados / 160

3.70112,3.658844,3.216056,2.08048,1.330663,1.374649,1.243986,1.307608,1.333745,1.32959,...,0.2184567,0.1930438,0.1836316,0.1474285,0.1416762,0.1183573,0.1091404,0.09394702,0.08213896,0.06865925


In [ ]:
minimo_vars = which.min(resultados)
minimo_vars

In [20]:
X[i,]

,V1,V3,V5,V6,V8,V10,V11,V12,V13,V17,...,V287,V288,V290,V291,V292,V294,V296,V297,V298,V300
48,99.80,86.00,81.600,66.400,56.400,42.00,40.200,40.000,42.00,70.600,...,261.40,264.40,252.00,213.60,196.60,174.80,150.00,127.20,114.80,105.20
17,122.00,108.20,101.000,76.800,66.200,51.00,40.600,44.400,41.00,65.200,...,311.60,287.80,265.20,235.40,225.80,178.20,163.40,130.20,133.60,111.20
112,127.40,112.20,96.000,77.800,60.200,61.00,48.000,41.000,48.20,65.600,...,259.00,269.00,232.60,234.00,225.20,190.00,149.60,136.80,136.40,103.60
111,130.00,123.80,102.400,91.800,60.800,46.80,49.600,47.000,51.40,81.000,...,262.80,266.40,229.20,228.60,211.40,173.40,142.00,135.00,129.00,108.00
4,146.00,133.75,108.500,107.250,76.000,44.50,46.750,49.250,52.50,70.750,...,297.00,287.50,289.00,249.25,240.00,201.25,165.25,155.75,132.50,118.00
110,134.50,117.00,95.250,89.500,58.250,42.75,50.500,43.750,55.25,66.000,...,273.00,262.75,252.75,224.50,227.75,170.75,156.25,136.50,122.25,111.75
149,321.00,304.60,256.800,218.600,153.600,111.00,99.000,88.000,87.20,120.800,...,222.80,227.60,221.00,201.00,194.20,167.40,147.20,142.00,131.40,108.60
15,140.80,141.60,118.000,92.400,75.000,53.40,45.600,41.800,43.80,50.600,...,226.40,224.40,205.40,185.80,184.80,163.40,129.40,121.40,121.20,114.60
89,132.33,118.67,91.333,83.333,59.333,48.00,55.667,46.667,56.00,76.333,...,274.33,247.67,249.33,233.33,212.67,165.67,160.00,136.33,131.00,117.67
21,54.20,53.60,46.600,49.000,31.200,38.80,33.000,36.400,34.40,40.400,...,154.80,153.80,145.00,144.00,137.40,124.20,114.40,112.00,99.20,98.40


In [26]:
df_train = X[-i,]
df_train$Y = TRAIN_Y[-i]
X_test = X[i,]
Y_test = TRAIN_Y[i]

model <- lm(Y~., data = df_train)
Y_pred = predict(model, X_test)
sum((Y_test - Y_pred)^2) / 16

[1] 0.007975499

In [28]:
df_train

,V1,V3,V5,V6,V8,V10,V11,V12,V13,V17,...,V288,V290,V291,V292,V294,V296,V297,V298,V300,Y
1,140.40,120.00,108.400,94.600,65.000,47.800,47.600,45.200,51.400,62.400,...,271.60,243.20,219.80,202.40,192.80,149.80,137.20,135.00,118.60,13.904
2,134.40,128.20,117.000,96.400,68.200,54.000,47.800,40.600,41.200,64.200,...,255.20,236.60,227.20,213.00,174.20,156.80,134.40,125.40,119.00,14.194
5,147.00,132.00,113.000,100.400,72.800,52.200,47.600,42.600,48.800,68.200,...,257.20,244.40,227.40,208.20,184.60,147.00,144.00,127.00,112.00,14.078
6,131.60,133.20,111.000,94.400,62.800,49.600,43.200,42.400,48.400,57.200,...,211.40,203.00,183.00,177.60,154.20,137.80,123.60,114.60,111.40,13.600
7,137.20,123.60,103.200,93.200,67.400,54.800,46.200,41.600,48.800,71.800,...,355.20,342.00,296.60,271.20,226.20,178.60,154.40,149.20,118.60,12.942
8,156.60,150.60,116.600,110.000,78.800,58.800,54.600,49.000,50.200,66.800,...,212.40,212.80,195.60,192.40,163.00,134.40,133.20,125.40,109.80,15.656
9,127.00,121.33,99.333,86.833,61.667,48.167,45.167,43.000,52.667,64.167,...,282.17,261.17,252.33,229.50,186.83,147.17,145.33,124.83,112.33,13.935
10,151.60,146.00,114.000,104.000,72.400,54.200,52.400,47.000,43.000,56.600,...,213.00,206.40,206.40,176.80,161.20,141.40,133.00,120.60,111.00,17.174
11,125.00,119.20,94.600,93.000,64.000,44.200,45.400,46.600,47.400,61.000,...,262.00,228.80,221.60,227.40,172.00,154.20,134.80,126.20,113.00,14.004
13,129.40,128.80,107.000,90.400,65.600,44.600,47.000,48.000,42.800,65.200,...,265.00,245.80,224.60,210.00,172.60,143.80,144.20,131.60,109.40,14.744


In [27]:
model


Call:
lm(formula = Y ~ ., data = df_train)

Coefficients:
(Intercept)           V1           V3           V5           V6           V8  
 24.6303280   -0.0152918    0.0201260    0.0291967   -0.0003297    0.0850896  
        V10          V11          V12          V13          V17          V18  
 -0.0064874    0.0228763   -0.0009206    0.0623054    0.0002155   -0.0335577  
        V19          V21          V24          V25          V27          V29  
 -0.0293646    0.0210764    0.0058088    0.0250471   -0.0574269   -0.0507220  
        V32          V36          V38          V42          V44          V46  
  0.0417474   -0.0486714    0.0050214   -0.0016033   -0.0148478   -0.0022329  
        V48          V50          V58          V59          V60          V61  
  0.0041504    0.0018243   -0.0068225    0.0042453    0.0016856    0.0056135  
        V62          V69          V70          V73          V74          V75  
 -0.0038974   -0.0016541   -0.0022217    0.0015370   -0.0049220    0.003

In [12]:
setSeed()
n = nrow(SELECTED_TRAIN_X)
indices = obtener_indices_kfold(n, 10)
X <- SELECTED_TRAIN_X
err = 0
for(i in indices){
    df_train = X[-i,]
    df_train$Y = TRAIN_Y[-i]
    
    X_test = X[i,]
    Y_test = TRAIN_Y[i]

    model <- lm(Y~., data = df_train)
    Y_pred = predict(model, X_test)
    err = err + sum((Y_test - Y_pred)^2)
}
err = err/n
print(paste("El error cuadratico medio con LOOCV de train es: ", err))

[1] "El error cuadratico medio con LOOCV de train es:  0.01627195439574"


In [11]:
model <- lm(SELECTED_TRAIN_DF$Y~as.matrix(SELECTED_TRAIN_X))
Y_pred <- predict(model, SELECTED_TRAIN_X)
sum((TRAIN_Y - Y_pred)^2) / 160

[1] 1.34711e-05

In [12]:
Y_pred

1          2          3          4          5          6          7 
13.9063232 14.1929556 14.6643649 14.7992163 14.0811809 13.5997286 12.9417230 
         8          9         10         11         12         13         14 
15.6583339 13.9381504 17.1723006 13.9985445 13.4443540 14.7407518 13.9782516 
        15         17         18         19         20         21         22 
17.0521503 13.9721361 14.3906753  6.0880557  6.5570175  4.8104509  6.8215995 
        23         24         25         26         28         29         30 
 5.1678728  3.5140576  8.1376063  5.2711811  6.2348708  6.0394757  5.9362667 
        31         32         33         34         35         36         37 
 6.2746395  5.3332608  7.3085289 14.9323160 16.1294428 16.2619885 12.9764783 
        39         40         42         43         44         45         46 
15.9149941 16.0150917 14.7996784 11.8290862 15.4871534 15.7542838 13.1867430 
        47         48         49         50         52         54         55 
14.0183559 12.9043298 15.5676705 12.9537109 17.0183515 15.6915635 15.8195835 
        56         57         59         60         61         62         63 
16.0150899  1.5990487  1.2185626  0.9876571  1.7020875  1.2826363  1.1785363 
        64         65         67         68         69         70         72 
 5.7689076  5.8829693  4.9122370  4.8151384  4.1042361  4.2111003  4.0651476 
        73         75         77         78         79         80         81 
 4.3393495  1.2131579 16.9573872 13.0923029 15.0430894 14.5101150 12.9174935 
        82         83         84         86         87         88         89 
14.6093113 17.5849125 15.0387545 14.2566321 13.4034954 14.3535725 14.2062384 
        90         91         92         93         94         95         96 
14.0944894 14.7828704 12.5802201 13.1660140 14.1608694 15.1664265 13.5810889 
        97         98         99        100        101        102        103 
13.8101489 13.6224253 13.6591307 15.0396825 15.5693006 11.7428364 13.1127282 
       104        105        106        108        109        110        111 
15.8488016 15.5745036 14.2623506 14.2090387  9.9178462 16.3471703 16.5526813 
       112        114        115        116        117        118        119 
15.3681766 15.8202217 14.3985967 14.9292302 14.0363436 15.0490568 14.1930792 
       120        123        124        125        126        127        129 
12.1385559 17.1286917 13.4396299 11.0204782 11.1655941 11.1735578 11.0616887 
       130        132        133        134        135        136        137 
15.9178320 16.4545103 17.1901476 16.4774228 11.5871357 13.1749135 14.7715705 
       138        139        140        141        142        143        144 
14.0858956 11.5534148 14.9760768 16.6524864 15.0227081 13.9499502 15.9438176 
       145        146        148        149        150        151        152 
15.7968137 14.9354417 16.0160442 13.4338904 14.4551029 15.3127932 12.1324332 
       153        155        156        157        158        159        160 
15.7320667 16.1926931 15.5544938 15.6520422 14.3721652 15.3980159 15.3316736 
       161        162        163        164        165        166        167 
15.4368204 15.1112695 15.1985260 15.9274879 12.6125426 15.1735820 13.9177513 
       168        169        170        171        172        173        174 
12.0476475 15.4786549 14.3789738 16.1286521 15.4693348 15.5218427 15.3661178 
       175        176        177        178        179        180 
11.8186954 12.1912082 12.2122419 15.8176646 12.4416568 13.8902399

In [13]:
TRAIN_Y

[1] 13.904 14.194 14.668 14.800 14.078 13.600 12.942 15.656 13.935 17.174
 [11] 14.004 13.440 14.744 13.984 17.050 13.976 14.389  6.084  6.558  4.808
 [21]  6.826  5.164  3.516  8.142  5.274  6.238  6.042  5.936  6.272  5.332
 [31]  7.306 14.930 16.128 16.256 12.976 15.912 16.016 14.800 11.826 15.490
 [41] 15.752 13.188 14.016 12.900 15.570 12.952 17.024 15.692 15.818 16.020
 [51]  1.602  1.220  0.986  1.700  1.282  1.180  5.772  5.886  4.910  4.812
 [61]  4.102  4.208  4.066  4.350  1.206 16.956 13.088 15.038 14.502 12.924
 [71] 14.606 17.586 15.040 14.258 13.402 14.355 14.207 14.093 14.782 12.576
 [81] 13.172 14.164 15.164 13.582 13.812 13.620 13.662 15.038 15.572 11.742
 [91] 13.110 15.854 15.576 14.266 14.206  9.922 16.345 16.550 15.370 15.820
[101] 14.400 14.926 14.030 15.050 14.202 12.138 17.125 13.440 11.012 11.168
[111] 11.174 11.060 15.918 16.452 17.190 16.476 11.590 13.178 14.774 14.084
[121] 11.552 14.974 16.654 15.033 13.950 15.954 15.802 14.930 16.010 13.432
[131] 14.456 15.308 12.134 15.726 16.196 15.556 15.654 14.374 15.388 15.328
[141] 15.438 15.114 15.202 15.928 12.606 15.170 13.912 12.044 15.480 14.388
[151] 16.130 15.470 15.522 15.366 11.826 12.196 12.208 15.816 12.446 13.890

In [ ]:
indices = obtener_indices_kfold(n, 10)
indices

In [ ]:
summary(model)

In [ ]:
resultados / 160

In [ ]:
plot(resultados[1,])

##### ANOTACION: 

* Borramos las otras variables que no estén seleccionadas en la iteración 140...

* Nos creamos un data set nuevo con esas 140... y ahora aplicamos forward (nos debería dar literalemtne lo mismo) pero sin el problema de la dimensión (ni correlación) y por ende obtendríamos los gráficos facheros para poner en el informe...


Vemos que para un índice de 16 (eso quiere decir, 16 variables seleccionadas para aplicar forward) se obtiene el valor más chico para el error con validación cruzada

In [ ]:
forw <- regsubsets(Y~., data = TRAIN_DF, method = "forward", nvmax=minimo_vars)

In [ ]:
SELECTED_TRAIN_X = TRAIN_DF[, names(which(summary(forw)$which[minimo_vars, ]))[2:(minimo_vars+1)]]
SELECTED_TRAIN_DF = SELECTED_TRAIN_X
SELECTED_TRAIN_DF$Y = TRAIN_Y

In [ ]:
colnames(SELECTED_TRAIN_X)

# Modelo 1: Regresion Lineal

In [ ]:
# No escalamos los datos porque no hay regularizacion.
modelo1 <- lm(Y~., data = SELECTED_TRAIN_DF)

Y_pred = predict(modelo1, SELECTED_TRAIN_X)
error <- sum((TRAIN_Y - Y_pred)^2) / nrow(SELECTED_TRAIN_X)
print(paste("El error cuadratico medio de train es: ", error))

In [ ]:
n = nrow(SELECTED_TRAIN_X)
indices = obtener_indices_kfold(n, n)
X <- SELECTED_TRAIN_X
err = 0
for(i in indices){
    df_train = X[-i,]
    df_train$Y = TRAIN_Y[-i]
    
    X_test = X[i,]
    Y_test = TRAIN_Y[i]

    model <- lm(Y~.,  data = df_train)
    Y_pred = predict(model, X_test)
    err = err + sum((Y_test - Y_pred)^2)
}
err = err/n
print(paste("El error cuadratico medio con LOOCV de train es: ", err))

In [ ]:
p = ncol(SELECTED_TRAIN_X)
num = n*err*(n-1)
den = sum((TRAIN_Y-mean(TRAIN_Y))^2)*(n-p)
hatY = 1-num/den
print(paste("R2 adj =  ", hatY))

# Modelo 2: Regresion Lineal Regularizada

In [ ]:
setSeed()
n = nrow(SELECTED_TRAIN_X)
k = 5
resultados <- data.frame(
  Alpha   =double(),
  Lambda  =double(),
  Escalador = integer(),
  Error   =double()
)

indices = obtener_indices_kfold(n, k)
X <- SELECTED_TRAIN_X

alphas <- seq(0, 1, 0.05)
lambdas <- logspace(-10, 3, 50)
escaladores <- c(StScaler(), MMScaler())


for (alpha in alphas) {
    for (lambda in lambdas) {
        for(s in 1:length(escaladores)){
            escalador = escaladores[[s]] 
            err = 0
            for (i in indices) {
                X_train_scaled = as.matrix(escalador$fit_transform(X[-i, ]))
                Y_train = TRAIN_Y[-i]

                X_test_scaled = as.matrix(escalador$transform(X[i, ]))
                Y_test = TRAIN_Y[i]

                model <- glmnet(X_train_scaled, Y_train, lambda=lambda, alpha=alpha)
                Y_pred = predict(model, X_test_scaled)
                err = err + sum((Y_test - Y_pred)^2)
            }
            resultados = rbind(resultados, list(alpha, lambda, s, err/n))
        }
    }
    message(progreso(alpha, alphas))
}

colnames(resultados)  <- c("alpha", "lambda", "Escalador", "Error")

In [ ]:
mejores_parametros <- resultados[which.min(resultados$Error),]
mejores_parametros

### Calculo del error por LOOCV

In [ ]:
n = nrow(SELECTED_TRAIN_X)
indices = obtener_indices_kfold(n, n)
X <- SELECTED_TRAIN_X
err = 0
escalador = escaladores[[mejores_parametros$Escalador]]
for(i in indices){
    X_train_scaled = as.matrix(escalador$fit_transform(X[-i, ]))
    Y_train = TRAIN_Y[-i]

    X_test_scaled = as.matrix(escalador$transform(X[i, ]))
    Y_test = TRAIN_Y[i]

    model <- glmnet(X_train_scaled, Y_train, lambda=mejores_parametros$lambda, alpha=mejores_parametros$alpha)
    Y_pred = predict(model, X_test_scaled)
    err = err + sum((Y_test - Y_pred)^2)
}
err = err/n
print(paste("El error cuadratico medio con LOOCV de train es: ", err))

In [ ]:
X_train_scaled <- as.matrix(escalador$fit_transform(X))
modelo2 <- glmnet(X_train_scaled, TRAIN_Y, lambda=mejores_parametros$lambda, alpha=mejores_parametros$alpha)

p = sum(modelo2$beta[,1] != 0)
num = n*err*(n-1)
den = sum((TRAIN_Y-mean(TRAIN_Y))^2)*(n-p)
hatY = 1-num/den
print(paste("R2 adj =  ", hatY))

# Modelo 4: árbol de regresión